# NB8 — Grounded VLM explanation V1

Chạy một case validation theo pipeline cố định: **frozen V5 scorer → LOO diagnosis → structured evidence → Qwen3-VL-4B-Instruct**.

Notebook này không train/fine-tune, không load test split và không đưa synthetic ground truth vào VLM. **Recommendation is not implemented** trong V1.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
BRANCH = "feat/vlm-explanation-v1"
REPO_ROOT = Path("/content/opisoverated")

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.colab import drive
drive.mount("/content/drive")

ARTIFACT_ROOT = Path("/content/drive/MyDrive/ML_Final")
os.environ["FASHION_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["FASHION_EMBEDDING_CACHE"] = str(ARTIFACT_ROOT / "fashionclip_item_embeddings.pt")
os.environ["FASHION_EMBEDDING_MANIFEST"] = str(ARTIFACT_ROOT / "embedding_manifest_v1.json")
os.environ["FASHION_CORE7_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "core7_drop_v2")
os.environ["FASHION_SCORER_READY_DIR"] = str(ARTIFACT_ROOT / "polyvore_core7_v2" / "scorer_ready_v2")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-vlm.txt")],
    check=True,
)

HEAD = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
STATUS = subprocess.check_output(["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True).strip()
assert not STATUS, f"Repository must be clean, got: {STATUS}"
print("Branch  :", BRANCH)
print("Git HEAD:", HEAD)
print("VLM     : Qwen/Qwen3-VL-4B-Instruct")

In [ ]:
# Fail fast before downloading the VLM weights.
for pattern in ("test_vlm*.py", "test_loo*.py", "test_scorer*.py"):
    test_run = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(test_run.stdout)
    if test_run.stderr:
        print(test_run.stderr)
    if test_run.returncode != 0:
        raise RuntimeError(f"Tests failed for {pattern}: {test_run.returncode}")
print("VLM + LOO + SCORER REGRESSION TESTS: PASS")

In [ ]:
import torch

from src.data.runtime_paths import load_runtime_paths
from src.diagnosis.loo import diagnose_outfit
from src.scorer.checkpoint import build_runtime_provenance, load_checkpoint
from src.scorer.dataset import CATEGORY_TO_ID, EmbeddingStore, build_dataset_from_runtime
from src.scorer.model import TypeAwarePairwiseScorer

assert torch.cuda.is_available(), "Select Runtime → Change runtime type → T4 GPU"
device = torch.device("cuda")
paths = load_runtime_paths(repo_root=REPO_ROOT)
provenance = build_runtime_provenance(paths, REPO_ROOT)
embedding_store = EmbeddingStore(paths.embedding_cache)
valid_dataset = build_dataset_from_runtime(paths, "valid", embedding_store=embedding_store)
assert len(valid_dataset) == 2284

BEST_PATH = REPO_ROOT / "artifacts" / "checkpoints" / "type_aware_pairwise_v1" / "final_val_auc_v5_seed42" / "best.pt"
assert BEST_PATH.is_file(), f"Missing canonical checkpoint: {BEST_PATH}"
payload = load_checkpoint(BEST_PATH, map_location="cpu", current_provenance=provenance)
model = TypeAwarePairwiseScorer.from_config(payload["config"])
model.load_state_dict(payload["model_state_dict"])
model.to(device).eval()

assert payload["epoch"] == 52
assert abs(float(payload["best_valid_roc_auc"]) - 0.6905082489625538) < 1e-12
print("Validation samples  :", len(valid_dataset))
print("Checkpoint epoch   :", payload["epoch"])
print("Checkpoint val AUC :", payload["best_valid_roc_auc"])

In [ ]:
# Deterministic demo selection uses only public input shape + label, never swap target.
demo_index = next(
    index
    for index, row in enumerate(valid_dataset.records)
    if row["label"] == 0 and len(row["items"]) >= 4
)
sample = valid_dataset[demo_index]
loo_result = diagnose_outfit(
    model,
    sample["item_embeddings"],
    sample["coarse_category_ids"],
    item_ids=sample["item_ids"],
)
ID_TO_CATEGORY = {value: key for key, value in CATEGORY_TO_ID.items()}
coarse_categories = [ID_TO_CATEGORY[int(value)] for value in sample["coarse_category_ids"].tolist()]

print("Demo sample       :", sample["sample_id"])
print("Item IDs          :", sample["item_ids"])
print("LOO problematic  :", loo_result["problematic_item_index"], loo_result["problematic_item_id"])
print("LOO deltas       :", [round(value, 5) for value in loo_result["deltas_without_minus_full"]])

In [ ]:
# Fetch exactly the selected validation item images from the source dataset.
from datasets import load_dataset
from PIL import Image

wanted_ids = set(sample["item_ids"])
source_items = load_dataset("codewaly/polyvore1000", "items", split="valid", streaming=True)
images_by_id = {}
for row in source_items:
    item_id = str(row["item_id"])
    if item_id in wanted_ids:
        image = row.get("image")
        if not isinstance(image, Image.Image):
            raise TypeError(f"Source image for {item_id} is not a PIL image")
        images_by_id[item_id] = image.convert("RGB")
        if len(images_by_id) == len(wanted_ids):
            break
missing_ids = sorted(wanted_ids - set(images_by_id))
assert not missing_ids, f"Missing source images: {missing_ids}"

IMAGE_DIR = Path("/content/vlm_item_images") / sample["sample_id"]
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
item_image_paths = []
for item_index, item_id in enumerate(sample["item_ids"]):
    image_path = IMAGE_DIR / f"item_{item_index:02d}.jpg"
    images_by_id[item_id].save(image_path, format="JPEG", quality=95)
    item_image_paths.append(image_path)
print("Fetched item images:", len(item_image_paths))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(item_image_paths), figsize=(3 * len(item_image_paths), 4))
if len(item_image_paths) == 1:
    axes = [axes]
for index, (axis, image_path) in enumerate(zip(axes, item_image_paths)):
    axis.imshow(Image.open(image_path))
    axis.set_title(f"{index}: {coarse_categories[index]}\n{sample['item_ids'][index]}")
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from src.vlm import build_vlm_evidence, load_vlm_config, VLMExplanationPipeline

evidence = build_vlm_evidence(
    loo_result,
    sample_id=sample["sample_id"],
    item_ids=sample["item_ids"],
    coarse_categories=coarse_categories,
)
serialized_evidence = json.dumps(evidence, ensure_ascii=False)
for forbidden_key in ("negative_metadata", "swapped_item_index", "target_swapped_item_index", "top1_correct"):
    assert forbidden_key not in serialized_evidence
assert evidence["recommendation"] == {"status": "not_implemented", "items": []}

CONFIG_PATH = REPO_ROOT / "configs" / "vlm_qwen3_vl_4b_instruct_v1.json"
vlm_config = load_vlm_config(CONFIG_PATH)
assert vlm_config["model"]["id"] == "Qwen/Qwen3-VL-4B-Instruct"
print(json.dumps(evidence, indent=2, ensure_ascii=False))

In [ ]:
# This cell is the real-model merge gate: it downloads 4B weights and runs Qwen inference.
import importlib.metadata

from src.vlm.qwen_backend import Qwen3VLBackend

backend = Qwen3VLBackend.from_config(vlm_config)
vlm_pipeline = VLMExplanationPipeline(backend, vlm_config)
vlm_run = vlm_pipeline.explain(evidence, item_image_paths)
assert type(backend).__name__ == "Qwen3VLBackend"
assert vlm_run["model_id"] == "Qwen/Qwen3-VL-4B-Instruct"
assert vlm_run["visual_analysis"]["schema_version"] == "vlm-visual-analysis-v1"
assert vlm_run["explanation"]["schema_version"] == "vlm-explanation-v1"
assert "headline" not in vlm_run["visual_analysis"]
assert "explanation" not in vlm_run["visual_analysis"]

RUN_DIR = ARTIFACT_ROOT / "vlm_runs" / "qwen3_vl_4b_instruct_v1" / sample["sample_id"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_PATH = RUN_DIR / "evidence.json"
RUN_PATH = RUN_DIR / "vlm_run.json"
SMOKE_PATH = RUN_DIR / "real_qwen_smoke_report.json"
smoke_report = {
    "schema_version": "vlm-real-qwen-smoke-v1",
    "status": "PASS",
    "git_head": HEAD,
    "split": "valid",
    "test_split_loaded": False,
    "sample_id": sample["sample_id"],
    "item_count": len(item_image_paths),
    "model_id": vlm_run["model_id"],
    "backend_class": type(backend).__name__,
    "cuda_device": torch.cuda.get_device_name(0),
    "checkpoint_epoch": int(payload["epoch"]),
    "evidence_sha256": vlm_run["evidence_sha256"],
    "generation_attempts": int(vlm_run["generation_attempts"]),
    "visual_analysis_schema": vlm_run["visual_analysis"]["schema_version"],
    "explanation_schema": vlm_run["explanation"]["schema_version"],
    "transformers_version": importlib.metadata.version("transformers"),
    "qwen_vl_utils_version": importlib.metadata.version("qwen-vl-utils"),
}
EVIDENCE_PATH.write_text(json.dumps(evidence, indent=2, ensure_ascii=False), encoding="utf-8")
RUN_PATH.write_text(json.dumps(vlm_run, indent=2, ensure_ascii=False), encoding="utf-8")
SMOKE_PATH.write_text(json.dumps(smoke_report, indent=2, ensure_ascii=False), encoding="utf-8")

print("CONSTRAINED VISUAL ANALYSIS")
print(json.dumps(vlm_run["visual_analysis"], indent=2, ensure_ascii=False))
print("DETERMINISTIC VIETNAMESE RENDER")
print(json.dumps(vlm_run["explanation"], indent=2, ensure_ascii=False))
print("Saved evidence:", EVIDENCE_PATH)
print("Saved VLM run :", RUN_PATH)
print("Saved smoke   :", SMOKE_PATH)
print("TEST SPLIT WAS NOT LOADED.")
print("[MERGE GATE] REAL QWEN SMOKE: PASS")

## Cách đọc kết quả

- `problematic_item_*` luôn do LOO quyết định; Qwen không được thay đổi.
- Qwen chỉ xuất enum trong `visual_analysis`; không có field free-text.
- Các câu trong `explanation` do code render từ template, không phải prose của Qwen.
- `visual_observations` là suy luận từ ảnh, không phải ground truth.
- Scorer logit không phải xác suất hay điểm đẹp khách quan.
- Recommendation is not implemented; V1 không đề xuất đồ thay thế.
- Nếu JSON sai contract, pipeline thử sửa đúng một lần rồi hard-fail.
- PR chỉ sẵn sàng merge sau khi cell cuối in `[MERGE GATE] REAL QWEN SMOKE: PASS` và smoke report được kiểm tra.